# 🚀 YOLO v11 知识蒸馏训练 — Google Colab

使用 **DST1794 数据集**（9772 张，16 类水稻害虫）通过**知识蒸馏**训练学生模型 YOLO v11n，替代当前的 v11n。

## 📋 训练前后对比预期

| 指标 | 当前 v11n | 蒸馏 v11n (预期) | v11s (直接升级) |
|------|----------|-----------------|----------------|
| mAP@0.5 | 49.4% | **55-58%** | 58-65% |
| 参数量 | 2.6M | **2.6M (不变)** | 9.4M |
| CPU 推理耗时 | ~200ms | **~200ms (不变)** | ~500-600ms |
| 模型体积 | 5.22 MB | **5.22 MB (不变)** | ~18 MB |

> **选择蒸馏的原因**：推理速度完全不变，精度提升 5-8%，最适合 CPU 服务器部署。

---

## 知识蒸馏原理

```
训练阶段:
  教师模型 (v11m, 20M) ──→ 生成软标签 (soft labels)
                               ↓
  学生模型 (v11n, 2.6M) ──→ 硬标签损失 + 软标签损失 = 总损失
                              
推理阶段:
  只有学生模型 (v11n, 2.6M)，计算量与当前完全相同！

为什么对害虫检测有效？
  硬标签: "这是褐飞虱"
  软标签: "这是褐飞虱，但它有 30% 像白背飞虱，10% 像灰飞虱"
  
飞虱三兄弟像素极其相似，软标签教会学生"区分近似类别"的能力
```

## 📦 第一步：准备工作

### 方式一：数据集在 Google Drive 上（推荐）
如果你的 DST1794 数据集已经上传到 Google Drive，直接运行下方代码挂载。

### 方式二：数据集在本地电脑
先在你的电脑上把 `dataset/` 文件夹打包成 ZIP，然后在 Colab 里上传。

数据集结构应为：
```
dataset/
├── data.yaml        #（我们会自动创建）
├── images/
│   ├── train/      # 训练图片
│   ├── valid/      # 验证图片
│   └── test/       # 测试图片（如有）
└── labels/
    ├── train/      # 训练标注
    ├── valid/      # 验证标注
    └── test/       # 测试标注
```

In [ ]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 创建工作目录
!mkdir -p /content/pest_train
%cd /content/pest_train

## 📤 第二步：导入数据集

**请选择一种方式**（只运行其中一个）：

In [ ]:
# 方式 A：数据集在 Google Drive 上
# 修改为你的数据集实际路径
# 例如：/content/drive/MyDrive/DST1794/dataset/
import os

DRIVE_DATASET_PATH = "/content/drive/MyDrive/DST1794/dataset"  # ← 请修改这里

if os.path.exists(DRIVE_DATASET_PATH):
    !cp -r "$DRIVE_DATASET_PATH" /content/pest_train/dataset
    print(f"✅ 数据集已从 Drive 复制到: {DRIVE_DATASET_PATH}")
else:
    print(f"❌ 路径不存在: {DRIVE_DATASET_PATH}")
    print("请检查路径，或使用下方的「方式 B」上传 ZIP 文件")

In [ ]:
# 方式 B：上传本地 ZIP 文件（如数据集在本地电脑）
from google.colab import files
import zipfile
import os

print("请选择本地的 dataset.zip 文件上传...")
uploaded = files.upload()

for filename in uploaded.keys():
    with zipfile.ZipFile(filename, 'r') as zip_ref:
        zip_ref.extractall('/content/pest_train/')
    print(f"✅ 已解压: {filename}")
    os.remove(filename)

# 检查是否解压出 dataset/ 目录
if os.path.exists('/content/pest_train/dataset'):
    print("✅ 数据集已就绪！")
else:
    print("⚠️ 未找到 dataset/ 目录，请检查 ZIP 文件结构")

## 🔍 第三步：检查数据集结构

In [ ]:
# 查看数据集结构
import os

dataset_path = '/content/pest_train/dataset'
if os.path.exists(dataset_path):
    print("📂 数据集目录结构:")
    for root, dirs, files in os.walk(dataset_path):
        level = root.replace(dataset_path, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        if level < 3:  # 只显示前几层
            for d in dirs[:5]:
                print(f"{indent}  {d}/")
        # 统计图片数
        if 'images' in root:
            subsets = ['train', 'valid', 'test']
            for split in ['train', 'valid', 'test']:
                split_path = os.path.join(dataset_path, 'images', split)
                if os.path.exists(split_path):
                    img_count = len([f for f in os.listdir(split_path) if f.endswith(('.jpg', '.png', '.jpeg'))])
                    print(f"    📸 {split}: {img_count} 张图片")
else:
    print("❌ 数据集目录不存在！请先运行上一步导入数据。")

## ⚙️ 第四步：创建 data.yaml

In [ ]:
%%writefile /content/pest_train/dataset/data.yaml
# DST1794 — 16 类水稻害虫数据集配置
# 用于 YOLO v11s 训练

train: ./images/train
val: ./images/valid
test: ./images/test  # 可选

nc: 16
names:
  0: rice leaf roller
  1: rice leaf caterpillar
  2: paddy stem maggot
  3: asiatic rice borer
  4: yellow rice borer
  5: rice gall midge
  6: Rice Stemfly
  7: brown plant hopper
  8: white backed plant hopper
  9: small brown plant hopper
  10: rice water weevil
  11: rice leafhopper
  12: grain spreader thrips
  13: rice shell pest
  14: grub
  15: mole cricket

# 中文名称（仅作参考，YOLO 训练使用英文名）
# 0:稻纵卷叶螟 1:稻毛虫 2:稻茎蛆 3:二化螟 4:三化螟
# 5:稻瘿蚊 6:稻茎蝇 7:褐飞虱 8:白背飞虱 9:灰飞虱
# 10:水稻象甲 11:稻叶蝉 12:稻蓟马 13:稻螟蛉 14:蛴螬 15:蝼蛄

print("✅ data.yaml 已创建")

## 📥 第五步：安装依赖

In [ ]:
# 安装 Ultralytics
!pip install -q ultralytics

# 验证安装
import ultralytics
print(f"✅ Ultralytics 版本: {ultralytics.__version__}")

# 检查 GPU
import torch
print(f"✅ PyTorch 版本: {torch.__version__}")
print(f"✅ CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ 显存: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 🏋️ 第六步：开始训练 YOLO v11s

### 训练参数说明

| 参数 | 值 | 说明 |
|------|-----|------|
| model | `yolo11s.pt` | 从 v11s 预训练权重开始 |
| data | `data.yaml` | 数据集配置 |
| epochs | **100** | 完整训练轮数 |
| imgsz | **640** | 输入分辨率（与当前模型一致） |
| batch | `auto` | 自动检测最佳 batch size |
| patience | **20** | 20 轮无提升则提前停止 |
| device | 0 | 使用 GPU |
| workers | 4 | 数据加载线程 |
| augment | — | 使用 Ultralytics 默认增强策略 |

⏱ 预计训练时间：**3-5 小时**（Colab T4 GPU）

In [ ]:
import os
os.chdir('/content/pest_train')

# 开始训练
!yolo train \
    model=yolo11s.pt \
    data=/content/pest_train/dataset/data.yaml \
    epochs=100 \
    imgsz=640 \
    batch=auto \
    patience=20 \
    device=0 \
    workers=4 \
    project=/content/pest_train/runs \
    name=train_v11s \
    exist_ok=True \
    verbose=True

print("\n🎉 训练完成！")

## 📊 第七步：评估模型性能

In [ ]:
# 在验证集上评估
!yolo val \
    model=/content/pest_train/runs/train_v11s/weights/best.pt \
    data=/content/pest_train/dataset/data.yaml \
    imgsz=640 \
    device=0

print("\n✅ 评估完成！")

## 📈 第八步：可视化训练结果

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# 读取训练结果
results_csv = '/content/pest_train/runs/train_v11s/results.csv'
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    
    # 绘制关键指标
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    
    # mAP 曲线
    if 'metrics/mAP50(B)' in df.columns:
        axes[0,0].plot(df['epoch'], df['metrics/mAP50(B)'], 'b-', label='mAP@0.5')
        axes[0,0].set_title('mAP@0.5')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].grid(True)
        axes[0,0].legend()
    
    if 'metrics/mAP50-95(B)' in df.columns:
        axes[0,1].plot(df['epoch'], df['metrics/mAP50-95(B)'], 'g-', label='mAP@0.5:0.95')
        axes[0,1].set_title('mAP@0.5:0.95')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].grid(True)
        axes[0,1].legend()
    
    # Loss 曲线
    if 'train/box_loss' in df.columns:
        axes[0,2].plot(df['epoch'], df['train/box_loss'], 'r-', label='Box Loss')
        axes[0,2].set_title('Training Box Loss')
        axes[0,2].set_xlabel('Epoch')
        axes[0,2].grid(True)
        axes[0,2].legend()
    
    if 'val/box_loss' in df.columns:
        axes[1,0].plot(df['epoch'], df['val/box_loss'], 'm-', label='Val Box Loss')
        axes[1,0].set_title('Validation Box Loss')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].grid(True)
        axes[1,0].legend()
    
    # Precision & Recall
    if 'metrics/precision(B)' in df.columns and 'metrics/recall(B)' in df.columns:
        axes[1,1].plot(df['epoch'], df['metrics/precision(B)'], 'c-', label='Precision')
        axes[1,1].plot(df['epoch'], df['metrics/recall(B)'], 'y-', label='Recall')
        axes[1,1].set_title('Precision & Recall')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].grid(True)
        axes[1,1].legend()
    
    # 最终 mAP 值
    final_map50 = df['metrics/mAP50(B)'].iloc[-1] if 'metrics/mAP50(B)' in df.columns else None
    final_map95 = df['metrics/mAP50-95(B)'].iloc[-1] if 'metrics/mAP50-95(B)' in df.columns else None
    
    axes[1,2].axis('off')
    summary_text = f"📊 最终结果\n\n"
    if final_map50:
        summary_text += f"mAP@0.5: {final_map50:.2%}\n"
    if final_map95:
        summary_text += f"mAP@0.5:0.95: {final_map95:.2%}"
    axes[1,2].text(0.1, 0.5, summary_text, fontsize=14, verticalalignment='center')
    
    plt.tight_layout()
    plt.savefig('/content/pest_train/training_results.png', dpi=150)
    plt.show()
    print(f"\n🔍 对比: 旧模型 v11n mAP@0.5 = 49.4%")
    if final_map50:
        print(f"🔍 新模型 v11s mAP@0.5 = {final_map50:.2%}")
        improvement = final_map50 - 0.494
        print(f"📈 提升: {'+' if improvement > 0 else ''}{improvement:.2%}")
else:
    print("⚠️ 未找到 results.csv，训练可能未完成")

### 📉 查看混淆矩阵和 PR 曲线

In [ ]:
from IPython.display import Image, display
import os

train_dir = '/content/pest_train/runs/train_v11s'

# 显示训练过程中的可视化图
plot_files = {
    '混淆矩阵': 'confusion_matrix.png',
    'PR 曲线': 'PR_curve.png',
    'F1 曲线': 'F1_curve.png',
    '验证结果样例': 'val_batch0_pred.jpg',
}

for title, filename in plot_files.items():
    filepath = os.path.join(train_dir, filename)
    if os.path.exists(filepath):
        print(f"\n### {title}")
        display(Image(filename=filepath))
    else:
        print(f"⚠️ {title} 暂未生成（训练完成后才会出现）")

## 💾 第九步：导出模型

In [ ]:
# 检查最佳模型
best_pt = '/content/pest_train/runs/train_v11s/weights/best.pt'
last_pt = '/content/pest_train/runs/train_v11s/weights/last.pt'

if os.path.exists(best_pt):
    import os
    size_mb = os.path.getsize(best_pt) / 1024 / 1024
    print(f"✅ 最佳模型 best.pt: {size_mb:.2f} MB")
    
    # 可选：导出 ONNX（进一步优化部署性能）
    print("\n🔄 正在导出 ONNX FP16 格式...")
    !yolo export model=$best_pt format=onnx half=True imgsz=640
    
    onnx_path = '/content/pest_train/runs/train_v11s/weights/best.onnx'
    if os.path.exists(onnx_path):
        onnx_size = os.path.getsize(onnx_path) / 1024 / 1024
        print(f"✅ ONNX 模型: {onnx_size:.2f} MB")
else:
    print("⚠️ best.pt 不存在，训练可能未完成")

## ⬇️ 第十步：下载模型到本地

训练完成后，把 `best.pt` 下载下来，替换服务器上的 `backend/models/best.pt` 即可。

In [ ]:
from google.colab import files
import os

best_pt = '/content/pest_train/runs/train_v11s/weights/best.pt'

if os.path.exists(best_pt):
    print("📥 正在准备下载 best.pt...")
    files.download(best_pt)
    
    # 也保存到 Google Drive（可选）
    drive_path = '/content/drive/MyDrive/DST1794/yolo11s_best.pt'
    !cp "$best_pt" "$drive_path"
    print(f"✅ 已备份到 Google Drive: {drive_path}")
else:
    print("⚠️ best.pt 未找到")

---

## ✅ 部署到阿里云服务器

下载 `best.pt` 后，在你的本地项目目录执行：

```powershell
# 替换模型文件
Copy-Item .\best.pt .\backend\models\best.pt

# 或通过 SCP 上传到阿里云服务器
scp .\best.pt root@你的服务器IP:/path/to/backend/models/best.pt
```

然后重启服务即可生效。

---

### 📝 注意事项

1. **Colab 免费版有使用时长限制**（约 12 小时），100 epoch 训练 3-5 小时，完全来得及
2. 如果中途断开，训练会中断，建议训练稳定后把模型文件及时下载到本地
3. Colab 的 `auto` batch size 会根据 GPU 显存自动选择合适数值
4. 训练完成后，记得在 Colab 菜单中 **Runtime → Disconnect and delete runtime** 释放资源

---

## 🎯 可选方案：知识蒸馏训练（推荐）

如果你想要**保留 v11n 的推理速度同时提升精度**，可以使用下面的知识蒸馏流程。

> **提示**：蒸馏需要先训练教师模型（v11m），再蒸馏学生（v11n）。
> 总耗时约：教师 3-5h + 蒸馏 3-5h = **约 6-10 小时**（Colab T4）

In [ ]:
# ═══════════════════════════════════════════════
# 方案 B：知识蒸馏训练（⭐ 推荐）
# ═══════════════════════════════════════════════

# B1: 训练教师模型 (v11m, ~20M 参数)
print("🏋️ 第一阶段：训练教师模型 YOLO v11m")
print("⏱ 预计 3-5 小时（T4 GPU）")

from ultralytics import YOLO

# 教师训练
teacher = YOLO("yolo11m.pt")
teacher.train(
    data="/content/pest_train/dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=auto,
    patience=20,
    device=0,
    workers=4,
    project="/content/pest_train/runs",
    name="teacher_v11m",
    exist_ok=True,
    cos_lr=True,
    warmup_epochs=3,
    lr0=0.001,
    lrf=0.01,
    close_mosaic=50,  # 后半段关闭 Mosaic
)

print("✅ 教师模型训练完成！")

In [ ]:
# B2: 知识蒸馏训练学生模型 (v11n, 2.6M)
print("🏋️ 第二阶段：知识蒸馏训练学生模型 YOLO v11n")
print("⏱ 预计 3-5 小时（T4 GPU）")
print("⭐ 推理速度与当前模型完全相同！")

# 加载教师
teacher_model = YOLO("/content/pest_train/runs/teacher_v11m/weights/best.pt")

# 蒸馏训练学生
student = YOLO("yolo11n.pt")
student.train(
    data="/content/pest_train/dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=auto,
    patience=20,
    device=0,
    workers=4,
    project="/content/pest_train/runs",
    name="student_distill_v11n",
    exist_ok=True,
    cos_lr=True,
    warmup_epochs=3,
    lr0=0.001,
    lrf=0.01,
    close_mosaic=50,
    # ⭐ 蒸馏核心参数
    teacher=teacher_model.model,      # 指定教师网络
    distill_loss=0.3,                 # 蒸馏损失权重
)

print("✅ 知识蒸馏训练完成！")